In [123]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

import numpy as np
import pandas as pd

from pathlib import Path

In [124]:
DATA_DIR = Path("../data/raw")
FILE_PATH = DATA_DIR / "fraudTest.csv"

dtf = pd.read_csv(FILE_PATH,index_col=0)

### features retained from EDA (distance_km to be computed) ###

In [126]:
numeric_features = [
    "amt",
    "city_pop",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "distance_km",
]

categorical_features = [
    "category",
    "merchant",
    "gender",
    "state",
    "job",
]

features = numeric_features + categorical_features
target = "is_fraud"

In [127]:
# Aversine formula

lat1 = np.radians(dtf["lat"])
lon1 = np.radians(dtf["long"])

lat2 = np.radians(dtf["merch_lat"])
lon2 = np.radians(dtf["merch_long"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

distance_km = 2 * 6371 * np.arcsin(np.sqrt(a))
dtf["distance_km"] = distance_km

### Prepare dataset split according to date of transaction (trans_date_trans_time) ###

**test dataset records after train dataset records**

In [128]:
dtf["trans_date_trans_time"] = pd.to_datetime(
    dtf["trans_date_trans_time"]
)

dtf = dtf.sort_values("trans_date_trans_time")

In [129]:
dtf["trans_date_trans_time"].min(), dtf["trans_date_trans_time"].max()

(Timestamp('2020-06-21 12:14:25'), Timestamp('2020-12-31 23:59:34'))

In [130]:
n = len(dtf)

train_end = int(n * 0.7)
val_end = int(n * 0.85)

train = dtf.iloc[:train_end].copy()
val = dtf.iloc[train_end:val_end].copy()
test = dtf.iloc[val_end:].copy()

### check ###

In [131]:
train["trans_date_trans_time"].max() < test["trans_date_trans_time"].min()

True

In [132]:
print(type(dtf.columns))
print(dtf.columns.values)
features = list(dtf.columns.values)
print(features)

<class 'pandas.Index'>
<ArrowStringArray>
['trans_date_trans_time',                'cc_num',              'merchant',
              'category',                   'amt',                 'first',
                  'last',                'gender',                'street',
                  'city',                 'state',                   'zip',
                   'lat',                  'long',              'city_pop',
                   'job',                   'dob',             'trans_num',
             'unix_time',             'merch_lat',            'merch_long',
              'is_fraud',           'distance_km']
Length: 23, dtype: str
['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'distance_km']


In [133]:
X_train = train[features]
y_train = train[target]

X_val = val[features]
y_val = val[target]

X_test = test[features]
y_test = test[target]

In [134]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

### fit ###

same as 	
X_train_transformed = preprocessor.fit_transform(X_train)

	classifier.fit(
		X_train_transformed,
		y_train
	)

In [42]:
logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['amt','city_pop','lat',...,'gender','state','job']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining colum

### validation ###

In [43]:
y_val_proba = logistic_model.predict_proba(X_val)[:, 1]

In [47]:
y_val_pred = (y_val_proba >= 0.5).astype(int)

In [46]:
print(type(y_val_proba))
print(y_val_proba[:2])

<class 'numpy.ndarray'>
[1.62291556e-05 1.29083013e-01]


In [48]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print(confusion_matrix(y_val, y_val_pred))

Precision : 0.0133
Recall    : 0.3534
F1-score  : 0.0255
[[75629  7446]
 [  183   100]]


### Explore diffferent thresholds ###

In [49]:
thresholds = np.arange(0.05, 0.96, 0.05)

results = []

for threshold in thresholds:
    y_pred = (y_val_proba >= threshold).astype(int)

    results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
    })

threshold_results = pd.DataFrame(results)

threshold_results

,threshold,precision,recall,f1
0,0.05,0.008677,0.568905,0.017094
1,0.10,0.009052,0.512367,0.017790
2,0.15,0.009293,0.473498,0.018228
3,0.20,0.009608,0.445230,0.018810
4,0.25,0.010036,0.427562,0.019613
5,0.30,0.010698,0.416961,0.020861
6,0.35,0.011289,0.402827,0.021963
7,0.40,0.012036,0.392226,0.023356
8,0.45,0.012375,0.367491,0.023944
9,0.50,0.013252,0.353357,0.025546


In [50]:
import plotly.express as px

fig = px.line(
    threshold_results,
    x="threshold",
    y=["precision", "recall", "f1"],
    markers=True,
)

fig.show()

In [52]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val,
    y_val_proba
)

fig = px.line(
    x=recall,
    y=precision,
    markers=True,
    labels={
        "x": "Recall",
        "y": "Precision"
    },
    title="Precision-Recall Curve"
)

fig.show()


In [53]:
idx = np.argmax(precision[:-1])

print("Precision :", precision[idx])
print("Recall    :", recall[idx])
print("Threshold :", thresholds[idx])

Precision : 0.11297071129707113
Recall    : 0.09540636042402827
Threshold : 0.9918910581676305


**On voit que le modèle Linear Regression donne une séparation limitée entre les deux classes. Le Recall est faible avec le seuil qui maximise la préçision**

### Modèle de type Random Forest Classifier ###

In [135]:
preprocessor_rf = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

In [136]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced",
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [137]:
rf_model.fit(X_train, y_train)

y_val_proba_rf = rf_model.predict_proba(X_val)[:, 1]

In [138]:
y_val_pred_rf = (y_val_proba_rf >= 0.5).astype(int)

In [139]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

precision = precision_score(y_val, y_val_pred_rf)
recall = recall_score(y_val, y_val_pred_rf)
f1 = f1_score(y_val, y_val_pred_rf)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print(confusion_matrix(y_val, y_val_pred_rf))

Precision : 0.7655
Recall    : 0.3922
F1-score  : 0.5187
[[83041    34]
 [  172   111]]


**Sur l'ensemble de validation et avec un seuil de 0,5, le Random Forest obtient une Precision et un F1 nettement supérieurs à ceux de la Logistic Regression, tout en améliorant légèrement le Recall. La réduction des faux positifs est très importante**

In [140]:
thresholds = np.arange(0.05, 0.96, 0.05)

results = []

for threshold in thresholds:
    y_pred = (y_val_proba_rf >= threshold).astype(int)

    results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
    })

threshold_results = pd.DataFrame(results)

threshold_results

,threshold,precision,recall,f1
0,0.05,0.136465,0.862191,0.235635
1,0.10,0.266979,0.805654,0.401055
2,0.15,0.392793,0.770318,0.520286
3,0.20,0.462791,0.703180,0.558205
4,0.25,0.550296,0.657244,0.599034
5,0.30,0.617021,0.614841,0.615929
6,0.35,0.646091,0.554770,0.596958
7,0.40,0.696078,0.501767,0.583162
8,0.45,0.732143,0.434629,0.545455
9,0.50,0.765517,0.392226,0.518692


In [141]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val,
    y_val_proba_rf
)

fig = px.line(
    x=recall,
    y=precision,
    markers=True,
    labels={
        "x": "Recall",
        "y": "Precision"
    },
    title="Precision-Recall Curve"
)

fig.show()

In [142]:
idx = np.argmax(precision[:-1])

print("Precision :", precision[idx])
print("Recall    :", recall[idx])
print("Threshold :", thresholds[idx])

Precision : 1.0
Recall    : 0.0176678445229682
Threshold : 0.8566666666666667


### Feature Engineering ###

In [143]:
dtf = dtf.sort_values("trans_date_trans_time").copy()

#### We want to consider historical card transactions history before a transaction ####

##### cc_num feature 1 : count of historical transactions for a cc_num #####

In [144]:
dtf["card_transaction_count"] = (
    dtf.groupby("cc_num").cumcount()
)

##### cc_num feature 2 : count of preceding frauds #####

In [145]:
dtf["card_fraud_count"] = (
    dtf.groupby("cc_num")["is_fraud"]
    .cumsum()
    - dtf["is_fraud"]
)

##### cc_num feature 3 : historical fraud rate #####

r = (F + alpha*p0)/(N+alpha)
- F = number of preceding frauds
- N = number of preceding transactions
- p0 = global fraud rate
- alpha = prior force

In [146]:
p0 = dtf["is_fraud"].mean()
alpha = 10

dtf["card_fraud_rate"] = (
    dtf["card_fraud_count"] + alpha * p0
) / (
    dtf["card_transaction_count"] + alpha
)

Check

In [147]:
dtf[
    [
        "trans_date_trans_time",
        "cc_num",
        "is_fraud",
        "card_transaction_count",
        "card_fraud_count",
        "card_fraud_rate",
    ]
].head(20)

,trans_date_trans_time,cc_num,is_fraud,card_transaction_count,card_fraud_count,card_fraud_rate
0,2020-06-21 12:14:25,2291163933867244,0,0,0,0.00386
1,2020-06-21 12:14:33,3573030041201292,0,0,0,0.00386
2,2020-06-21 12:14:53,3598215285024754,0,0,0,0.00386
3,2020-06-21 12:15:15,3591919803438423,0,0,0,0.00386
4,2020-06-21 12:15:17,3526826139003047,0,0,0,0.00386
5,2020-06-21 12:15:37,30407675418785,0,0,0,0.00386
6,2020-06-21 12:15:44,213180742685905,0,0,0,0.00386
7,2020-06-21 12:15:50,3589289942931264,0,0,0,0.00386
8,2020-06-21 12:16:10,3596357274378601,0,0,0,0.00386
9,2020-06-21 12:16:11,3546897637165774,0,0,0,0.00386


In [148]:
dtf.loc[
    dtf["cc_num"] == 4725837176265195730,
    [
        "trans_date_trans_time",
        "cc_num",
        "is_fraud",
        "card_transaction_count",
        "card_fraud_count",
        "card_fraud_rate",
    ]
]


,trans_date_trans_time,cc_num,is_fraud,card_transaction_count,card_fraud_count,card_fraud_rate
4562,2020-06-22 16:34:45,4725837176265195730,0,0,0,0.003860
5416,2020-06-22 20:35:51,4725837176265195730,0,1,0,0.003509
16225,2020-06-26 23:36:15,4725837176265195730,0,2,0,0.003217
21751,2020-06-28 22:22:53,4725837176265195730,0,3,0,0.002969
23113,2020-06-29 07:36:04,4725837176265195730,0,4,0,0.002757
...,...,...,...,...,...,...
550011,2020-12-30 06:30:45,4725837176265195730,0,237,16,0.064934
551161,2020-12-30 16:16:26,4725837176265195730,0,238,16,0.064672
554327,2020-12-31 16:04:19,4725837176265195730,0,239,16,0.064412
554655,2020-12-31 17:48:13,4725837176265195730,0,240,16,0.064154


In [149]:
card_features = [
    "card_transaction_count",
    "card_fraud_count",
    "card_fraud_rate",
]

dtf[card_features].describe()

,card_transaction_count,card_fraud_count,card_fraud_rate
count,555719.000000,555719.000000,555719.000000
mean,390.239182,1.283053,0.003999
std,292.891642,3.548173,0.015318
min,0.000000,0.000000,0.000026
25%,152.000000,0.000000,0.000076
50%,332.000000,0.000000,0.000150
75%,575.000000,0.000000,0.000460
max,1473.000000,19.000000,0.566896


In [150]:
dtf[
    ["cc_num", "is_fraud"] + card_features
].head(10)

,cc_num,is_fraud,card_transaction_count,card_fraud_count,card_fraud_rate
0,2291163933867244,0,0,0,0.00386
1,3573030041201292,0,0,0,0.00386
2,3598215285024754,0,0,0,0.00386
3,3591919803438423,0,0,0,0.00386
4,3526826139003047,0,0,0,0.00386
5,30407675418785,0,0,0,0.00386
6,213180742685905,0,0,0,0.00386
7,3589289942931264,0,0,0,0.00386
8,3596357274378601,0,0,0,0.00386
9,3546897637165774,0,0,0,0.00386


In [151]:
features_to_drop = [
    "is_fraud",
    "cc_num",
    "trans_num",
    "trans_date_trans_time",
]

X = dtf.drop(columns=features_to_drop)
y = dtf["is_fraud"]

In [152]:
n = len(dtf)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

In [153]:
numeric_features = numeric_features +card_features
print('numeric features')
print(numeric_features)
print('categorical_features')
print(categorical_features)
print

numeric features
['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long', 'distance_km', 'card_transaction_count', 'card_fraud_count', 'card_fraud_rate']
categorical_features
['category', 'merchant', 'gender', 'state', 'job']


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [154]:
preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

In [155]:
rf_model_card = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced",
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [156]:
rf_model_card.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['merchant','category','amt',...,'card_transaction_count', 'card_fraud_count','card_fraud_rate']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``r

In [157]:
y_val_proba_rf_card = (
    rf_model_card.predict_proba(X_val)[:, 1]
)

In [158]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

threshold = 0.5

y_val_pred_rf_card = (
    y_val_proba_rf_card >= threshold
).astype(int)

print("Precision :", precision_score(y_val, y_val_pred_rf_card))
print("Recall    :", recall_score(y_val, y_val_pred_rf_card))
print("F1        :", f1_score(y_val, y_val_pred_rf_card))
print()
print(confusion_matrix(y_val, y_val_pred_rf_card))

Precision : 0.95625
Recall    : 0.5406360424028268
F1        : 0.690744920993228

[[83068     7]
 [  130   153]]


Avec l'ajout des 3 features, au seuil 0.5 on a amélioré le f1-score, le nombre de TP est en hausse et le nombre de FP a été divisé par 5.
---------------------------------- métriques du précédent modèle ----------------------------------
 
Precision : 0.7655
Recall    : 0.3922
F1-score  : 0.5187
[[83041    34]
 [  172   111]]

In [160]:
thresholds_to_test = np.arange(0.05, 1.00, 0.05)

results = []

for threshold in thresholds_to_test:
    y_pred = (
        y_val_proba_rf_card >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred
    ).ravel()

    results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    })

threshold_results_card = pd.DataFrame(results)

threshold_results_card

,threshold,precision,recall,f1,TP,FP,FN,TN
0,0.05,0.050952,0.964664,0.096791,273,5085,10,77990
1,0.10,0.148770,0.939929,0.256881,266,1522,17,81553
2,0.15,0.265608,0.886926,0.408795,251,694,32,82381
3,0.20,0.387521,0.833922,0.529148,236,373,47,82702
4,0.25,0.523342,0.752650,0.617391,213,194,70,82881
5,0.30,0.668942,0.692580,0.680556,196,97,87,82978
6,0.35,0.818584,0.653710,0.726916,185,41,98,83034
7,0.40,0.857843,0.618375,0.718686,175,29,108,83046
8,0.45,0.925287,0.568905,0.704595,161,13,122,83062
9,0.50,0.956250,0.540636,0.690745,153,7,130,83068


On a le meilleur f1-score vers le seuil de 0.35

In [89]:

precision, recall, thresholds = precision_recall_curve(
    y_val,
    y_val_proba_rf_card
)

fig = px.line(
    x=recall,
    y=precision,
    markers=True,
    labels={
        "x": "Recall",
        "y": "Precision"
    },
    title="Precision-Recall Curve"
)

fig.show()

##### cc_num feature 4 : card_avg_amount as-of #####

In [161]:
dtf["card_amount_sum"] = (
    dtf.groupby("cc_num")["amt"].cumsum()
    - dtf["amt"]
)

In [162]:
dtf["card_avg_amount"] = (
    dtf["card_amount_sum"]
    / dtf["card_transaction_count"]
)

##### check #####

In [163]:
card = dtf["cc_num"].iloc[0]

dtf.loc[
    dtf["cc_num"] == card,
    [
        "trans_date_trans_time",
        "amt",
        "card_transaction_count",
        "card_amount_sum",
        "card_avg_amount",
    ]
].head(20)

,trans_date_trans_time,amt,card_transaction_count,card_amount_sum,card_avg_amount
0,2020-06-21 12:14:25,2.86,0,0.00,NaN
606,2020-06-21 15:39:04,17.15,1,2.86,2.860000
4324,2020-06-22 15:24:57,89.55,2,20.01,10.005000
4340,2020-06-22 15:29:43,24.67,3,109.56,36.520000
4590,2020-06-22 16:43:19,6.28,4,134.23,33.557500
5196,2020-06-22 19:32:24,5.35,5,140.51,28.102000
5979,2020-06-22 23:21:29,12.52,6,145.86,24.310000
6424,2020-06-23 02:38:21,5.82,7,158.38,22.625714
6698,2020-06-23 04:49:53,4.04,8,164.20,20.525000
7250,2020-06-23 08:57:06,67.68,9,168.24,18.693333


In [164]:
card_features = card_features + ["card_avg_amount"]

In [96]:
dtf["card_avg_amount"].isna().sum()

np.int64(924)

In [165]:
numeric_features = numeric_features + ["card_avg_amount"]

In [166]:
print(numeric_features)

['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long', 'distance_km', 'card_transaction_count', 'card_fraud_count', 'card_fraud_rate', 'card_avg_amount']


In [167]:
# dtf a été trié selon trans_date_trans_time asc
dtf_model = dtf.copy()

In [168]:
columns_to_drop = [
    "is_fraud",              # target
    "cc_num",                # utilisé uniquement pour créer les features historiques
    "trans_num",             # identifiant
    "trans_date_trans_time", # pas disponible à l'inférence
]

In [169]:
X = dtf_model.drop(columns=columns_to_drop)
y = dtf_model["is_fraud"]

In [170]:
numeric_features

['amt',
 'city_pop',
 'lat',
 'long',
 'merch_lat',
 'merch_long',
 'distance_km',
 'card_transaction_count',
 'card_fraud_count',
 'card_fraud_rate',
 'card_avg_amount']

In [171]:
categorical_features

['category', 'merchant', 'gender', 'state', 'job']

In [172]:
features = numeric_features + categorical_features

X = dtf_model[features]
y = dtf_model["is_fraud"]

In [173]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [174]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

In [175]:
categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [176]:
preprocessor_rf = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
    ]
)

In [177]:
from sklearn.ensemble import RandomForestClassifier
rf_model_card = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced",
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [178]:
n = len(dtf_model)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

In [180]:
rf_model_card.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['amt','city_pop','lat',...,'gender','state','job']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining colum

In [181]:
y_val_proba_rf_card = (
    rf_model_card.predict_proba(X_val)[:, 1]
)

In [182]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

threshold = 0.5

y_val_pred_rf_card = (
    y_val_proba_rf_card >= threshold
).astype(int)

print("Precision :", precision_score(y_val, y_val_pred_rf_card))
print("Recall    :", recall_score(y_val, y_val_pred_rf_card))
print("F1        :", f1_score(y_val, y_val_pred_rf_card))
print()
print(confusion_matrix(y_val, y_val_pred_rf_card))

Precision : 0.9605263157894737
Recall    : 0.5159010600706714
F1        : 0.671264367816092

[[83069     6]
 [  137   146]]


*Par rapport au modèle précédent, au seuil par défaut de 0.5 :
- 1 FP de moins
- mais 7 TP de moins 
- recall baisse de 2.5 points
- F1 passe de 0.691 à 0.671*

---------------------------- modèle précédent (sans feature 4) au seuil 0.5 ----------------------

Precision : 0.95625
Recall    : 0.5406360424028268
F1        : 0.690744920993228

[[83068     7]
 [  130   153]]

In [183]:
thresholds_to_test = np.arange(0.05, 1.00, 0.05)

results = []

for threshold in thresholds_to_test:
    y_pred = (
        y_val_proba_rf_card >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred
    ).ravel()

    results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    })

threshold_results_card = pd.DataFrame(results)

threshold_results_card

,threshold,precision,recall,f1,TP,FP,FN,TN
0,0.05,0.049578,0.954064,0.094257,270,5176,13,77899
1,0.10,0.156805,0.936396,0.268626,265,1425,18,81650
2,0.15,0.283296,0.886926,0.429427,251,635,32,82440
3,0.20,0.426259,0.837456,0.564958,237,319,46,82756
4,0.25,0.592992,0.777385,0.672783,220,151,63,82924
5,0.30,0.724382,0.724382,0.724382,205,78,78,82997
6,0.35,0.801688,0.671378,0.730769,190,47,93,83028
7,0.40,0.858586,0.600707,0.706861,170,28,113,83047
8,0.45,0.906977,0.551237,0.685714,156,16,127,83059
9,0.50,0.960526,0.515901,0.671264,146,6,137,83069


|                      | Ancien RF + 3 features carte | Nouveau RF + `card_avg_amount` |
| -------------------- | ---------------------------: | -----------------------------: |
| **F1 @ 0.50**        |                       69.07% |                         67.13% |
| **F1 @ 0.40**        |                       71.87% |                         70.69% |
| **F1 @ 0.35**        |                       72.69% |                     **73.08%** |
| **Precision @ 0.35** |                       81.86% |                         80.17% |
| **Recall @ 0.35**    |                       65.37% |                     **67.14%** |


On garde le dernier modèle. L'exploration des seuils montre un F1-score max de 73.08% obtenu pour un seuil autour de 0.35 contre 72.69% pour le modèle précédent.
Pour la suite on considère qu'une décision business est de retenir un seuil autour de 0.35-0.40.

In [185]:
for name, y in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test),
]:
    print(f"{name}: {y.mean():.3%}")

Train: 0.457%
Validation: 0.339%
Test: 0.103%


**On peut voir ci-dessus que la proportion de fraudes dans l'échantillon varie notablement entre les 3 sous-ensembles et donc au cours du temps. On reviendra sur ce point après le modèle RFC avec une 4éme feature calculée. Possible dérive du dataset, possible baisse des performances**

## drift check ###

In [193]:
dtf_drift_analysis = dtf_model.copy()
dtf_drift_analysis.columns

Index(['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt',
       'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud', 'distance_km', 'card_transaction_count',
       'card_fraud_count', 'card_fraud_rate', 'card_amount_sum',
       'card_avg_amount'],
      dtype='str')

#### Evolution du taux de fraude en fonction du mois et du dataset ####

In [194]:

dtf_drift_analysis = dtf_drift_analysis.sort_values("trans_date_trans_time").reset_index(drop=True)

# splits
n = len(dtf_drift_analysis)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

dtf_drift_analysis["split"] = "test"
dtf_drift_analysis.loc[:train_end - 1, "split"] = "train"
dtf_drift_analysis.loc[train_end:val_end - 1, "split"] = "validation"

# taux de fraude mensuel

dtf_drift_analysis["year"] = dtf_drift_analysis["trans_date_trans_time"].dt.year
dtf_drift_analysis["month"] = dtf_drift_analysis["trans_date_trans_time"].dt.month
dtf_drift_analysis["year_month"] = (
    dtf_drift_analysis["trans_date_trans_time"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly_fraud = (
    dtf_drift_analysis.groupby("year_month")
    .agg(
        fraud_rate=("is_fraud", "mean"),
        transaction_count=("is_fraud", "count"),
    )
    .reset_index()
)

train_end_date = dtf_drift_analysis.loc[train_end, "trans_date_trans_time"]
val_end_date = dtf_drift_analysis.loc[val_end, "trans_date_trans_time"]

In [195]:
split_sizes = (
    dtf_drift_analysis["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

split_summary = (
    dtf_drift_analysis.groupby("split")
    .agg(
        transaction_count=("is_fraud", "size"),
        fraud_count=("is_fraud", "sum"),
        fraud_rate=("is_fraud", "mean"),
        start_date=("trans_date_trans_time", "min"),
        end_date=("trans_date_trans_time", "max"),
    )
    .reindex(["train", "validation", "test"])
)

print(split_summary)

            transaction_count  fraud_count  fraud_rate          start_date  \
split                                                                        
train                  389003         1776    0.004566 2020-06-21 12:14:25   
validation              83358          283    0.003395 2020-11-20 18:26:42   
test                    83358           86    0.001032 2020-12-13 20:01:56   

                      end_date  
split                           
train      2020-11-20 18:25:35  
validation 2020-12-13 20:01:30  
test       2020-12-31 23:59:34  


In [196]:
import pandas as pd
import plotly.express as px

fig = px.line(
    monthly_fraud,
    x="year_month",
    y="fraud_rate",
    markers=True,
    title="Évolution du taux de fraude — Train / Validation / Test",
    labels={
        "year_month": "Mois",
        "fraud_rate": "Taux de fraude",
    },
)

fig.update_yaxes(tickformat=".2%")


# Frontière train / validation
fig.add_vline(
    x=train_end_date,
    line_dash="dash",
    annotation_text="Train → Validation",
    annotation_position="top",
)


# Frontière validation / test
fig.add_vline(
    x=val_end_date,
    line_dash="dash",
    annotation_text="Validation → Test",
    annotation_position="top",
)

fig.show()

In [197]:
fraud_by_month_split = (
    dtf_drift_analysis.groupby(["year_month", "split"])["is_fraud"]
    .agg(
        transaction_count="count",
        fraud_count="sum",
        fraud_rate="mean",
    )
    .reset_index()
)

print(fraud_by_month_split)

  year_month       split  transaction_count  fraud_count  fraud_rate
0 2020-06-01       train              30058          133    0.004425
1 2020-07-01       train              85848          321    0.003739
2 2020-08-01       train              88759          415    0.004676
3 2020-09-01       train              69533          340    0.004890
4 2020-10-01       train              69348          384    0.005537
5 2020-11-01       train              45457          183    0.004026
6 2020-11-01  validation              27178          111    0.004084
7 2020-12-01        test              83358           86    0.001032
8 2020-12-01  validation              56180          172    0.003062


In [199]:
monthly_fraud

,year_month,fraud_rate,transaction_count
0,2020-06-01,0.004425,30058
1,2020-07-01,0.003739,85848
2,2020-08-01,0.004676,88759
3,2020-09-01,0.004890,69533
4,2020-10-01,0.005537,69348
5,2020-11-01,0.004048,72635
6,2020-12-01,0.001849,139538


**On constate une baisse du taux de fraude marquée en décembre et aussi un nombre de transactions plus élevé en décembre

la frontière du split validation -> test a lieu en décembre (le 13) et le taux de fraude varie fortement engtre validation et test**

In [202]:
print(type(y_val.to_numpy()))

y_val_proba_rf_card = (
    rf_model_card.predict_proba(X_val)[:, 1]
)

print(type(y_val_proba_rf_card))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [204]:
y_val_proba_rf_card = (
    rf_model_card.predict_proba(X_val)[:, 1]
)

y_test_proba_rf_card =(
    rf_model_card.predict_proba(X_test)[:, 1]
)

THRESHOLD = 0.35


y_val_pred = (
        y_val_proba_rf_card >= THRESHOLD
).astype(int)

y_test_pred = (
    y_test_proba_rf_card >= THRESHOLD
)

def compute_metrics(split_type:str,y_pred: np.ndarray, y:np.ndarray,threshold:float, result:list):
    tn, fp, fn, tp = confusion_matrix(
            y,
            y_pred
        ).ravel()

    results.append({
        "set":split_type,
        "threshold": threshold,
        "precision": precision_score(
            y, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y, y_pred, zero_division=0
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    })

results = []
compute_metrics("val",y_val_pred,y_val.to_numpy(),THRESHOLD, results)
compute_metrics("test",y_test_pred,y_test.to_numpy(),THRESHOLD, results)

val_test_metrics = pd.DataFrame(results)

val_test_metrics

,set,threshold,precision,recall,f1,TP,FP,FN,TN
0,val,0.35,0.79668,0.678445,0.732824,192,49,91,83026
1,test,0.35,0.52500,0.732558,0.611650,63,57,23,83215


**la prévalence du taux de fraude chute fortement + la precision chute fortement + le recall reste élevé.
La baisse de prévalence pourrait expliquer une partie de la baisse de precision**

#### Vérification de l'évolution des features ####

In [205]:
comparison = (
    dtf_drift_analysis.groupby("split")
    .agg(
        avg_amt=("amt", "mean"),
        median_amt=("amt", "median"),
        avg_distance=("distance_km", "mean"),
        avg_card_transactions=("card_transaction_count", "mean"),
        avg_card_fraud_rate=("card_fraud_rate", "mean"),
    )
    .reindex(["train", "validation", "test"])
)

print(comparison)

              avg_amt  median_amt  avg_distance  avg_card_transactions  \
split                                                                    
train       69.524650      47.330     76.110918             272.871425   
validation  69.102917      47.530     75.995777             604.657237   
test        69.067454      46.945     76.185953             723.535917   

            avg_card_fraud_rate  
split                            
train                  0.004012  
validation             0.004113  
test                   0.003822  


#### Nombre de transactions, de fraudes et taux de fraude par catégorie d'achat pour chaque split ####

In [208]:
category_split = (
    dtf_drift_analysis.groupby(["split", "category"])["is_fraud"]
    .agg(
        transaction_count="count",
        fraud_count="sum",
        fraud_rate="mean",
    )
    .reset_index()
)

category_split.sort_values(by=["category", "fraud_count"], ascending=[True,True])

,split,category,transaction_count,fraud_count,fraud_rate
0,test,entertainment,6000,0,0.000000
28,validation,entertainment,6033,7,0.001160
14,train,entertainment,28071,52,0.001852
1,test,food_dining,5743,1,0.000174
29,validation,food_dining,5797,5,0.000863
15,train,food_dining,27728,48,0.001731
2,test,gas_transport,8417,9,0.001069
30,validation,gas_transport,8640,23,0.002662
16,train,gas_transport,39313,122,0.003103
3,test,grocery_net,2945,1,0.000340


**On observe que le nombre de fraudes diminue dans presque toutes les catégories entre train, val, test**

#### Répartition du nombre de transactions par catégorie pour chaque échantillon ####

In [210]:
category_distribution = (
    dtf_drift_analysis.groupby(["split", "category"])
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum())
    .rename("share")
    .reset_index()
)

category_distribution.sort_values(by=["split","share"], ascending=[True,False])

,split,category,share
2,test,gas_transport,0.100974
6,test,home,0.095972
4,test,grocery_pos,0.092181
12,test,shopping_pos,0.091581
7,test,kids_pets,0.088054
11,test,shopping_net,0.074510
0,test,entertainment,0.071979
10,test,personal_care,0.071919
1,test,food_dining,0.068896
5,test,health_fitness,0.066376


**On peut voir que la répartition du nombre de transactions par category évolue peut selon l'échantillon**

In [211]:
fraud_rate_by_category = (
    dtf_drift_analysis.groupby(["split", "category"])["is_fraud"]
    .mean()
    .reset_index(name="fraud_rate")
)

fraud_rate_by_category

,split,category,fraud_rate
0,test,entertainment,0.000000
1,test,food_dining,0.000174
2,test,gas_transport,0.001069
3,test,grocery_net,0.000340
4,test,grocery_pos,0.001822
5,test,health_fitness,0.000181
6,test,home,0.000625
7,test,kids_pets,0.000409
8,test,misc_net,0.003222
9,test,misc_pos,0.000000


#### Relation montant -> fraude selon l'échantillon ####

In [213]:
amount_bins = [0, 10, 25, 50, 100, 250, 500, 1000, float("inf")]

dtf_drift_analysis["amt_bin"] = pd.cut(
    dtf_drift_analysis["amt"],
    bins=amount_bins,
    right=False
)

fraud_rate_by_amount = (
    dtf_drift_analysis.groupby(
        ["split", "amt_bin"],
        observed=True
    )["is_fraud"]
    .mean()
    .reset_index(name="fraud_rate")
)

fraud_rate_by_amount

,split,amt_bin,fraud_rate
0,test,"[0.0, 10.0)",0.000323
1,test,"[10.0, 25.0)",0.001202
2,test,"[25.0, 50.0)",0.000000
3,test,"[50.0, 100.0)",0.000000
4,test,"[100.0, 250.0)",0.000397
5,test,"[250.0, 500.0)",0.010807
6,test,"[500.0, 1000.0)",0.067524
7,test,"[1000.0, inf)",0.034483
8,train,"[0.0, 10.0)",0.001189
9,train,"[10.0, 25.0)",0.006683


**plus le montant est élevé, plus le taux de fraude est élevé dans tous les splits, mais la relation s'affaiblit fortement dans le temps**

En résumé

- distribution de category env stable
- distribution de amt env stable
- distribution de distance_km env stable
- Le taux de fraude baisse fortement
- même conditionnellement au montant le taux de fraude diminue
- ce qui laisse penser P(Y|X) change et pas seulement P(X)

In [214]:
# Probabilités prédites
val_proba = rf_model_card.predict_proba(X_val)[:, 1]
test_proba = rf_model_card.predict_proba(X_test)[:, 1]

In [215]:
for name, y_proba in [
    ("validation", val_proba),
    ("test", test_proba),
]:
    print(name)
    print("Nombre d'alertes :", (y_proba >= 0.35).sum())
    print("Taux d'alerte    :", (y_proba >= 0.35).mean())

validation
Nombre d'alertes : 241
Taux d'alerte    : 0.002891144221310492
test
Nombre d'alertes : 120
Taux d'alerte    : 0.0014395738861297056


In [120]:
import plotly.express as px

scores = pd.DataFrame({
    "validation": val_proba,
    "test": test_proba,
})

fig = px.histogram(
    scores,
    x=["validation", "test"],
    barmode="overlay",
    nbins=50,
    histnorm="probability",
    title="Distribution des probabilités prédites",
    labels={
        "value": "Probabilité de fraude",
        "variable": "Split",
    },
)

fig.add_vline(
    x=0.35,
    line_dash="dash",
    annotation_text="Seuil = 0.35",
)

fig.show()

In [121]:
import pandas as pd

def calibration_table(y_true, probabilities, n_bins=10):

    df = pd.DataFrame({
        "y_true": y_true,
        "probability": probabilities,
    })

    df["bin"] = pd.qcut(
        df["probability"],
        q=n_bins,
        duplicates="drop",
    )

    calibration = (
        df.groupby("bin", observed=True)
        .agg(
            mean_predicted_probability=("probability", "mean"),
            actual_fraud_rate=("y_true", "mean"),
            transaction_count=("y_true", "size"),
        )
        .reset_index()
    )

    return calibration

In [122]:
val_calibration = calibration_table(
    y_val,
    val_proba,
)

test_calibration = calibration_table(
    y_test,
    test_proba,
)

print("Validation")
print(val_calibration)

print("\nTest")
print(test_calibration)

Validation
                 bin  mean_predicted_probability  actual_fraud_rate  \
0  (-0.001, 0.00333]                    0.000335           0.000047   
1    (0.00333, 0.01]                    0.007997           0.000000   
2     (0.01, 0.0333]                    0.021983           0.000818   
3    (0.0333, 0.937]                    0.090696           0.033919   

   transaction_count  
0              63544  
1               4400  
2               7336  
3               8078  

Test
                 bin  mean_predicted_probability  actual_fraud_rate  \
0  (-0.001, 0.00333]                    0.000408           0.000019   
1  (0.00333, 0.0167]                    0.011081           0.000000   
2   (0.0167, 0.0333]                    0.026405           0.000240   
3   (0.0333, 0.0533]                    0.044681           0.000000   
4    (0.0533, 0.917]                    0.095551           0.010053   

   transaction_count  
0              52309  
1               7204  
2               